# GxP-LLM Fine-Tuning with Unsloth + TRL

Run on Kaggle GPU (T4/P100). Self-contained: installs deps, loads data, trains, logs to W&B.

In [ ]:
# Install dependencies
%pip install -q torch==2.5.1 transformers==4.46.3 trl==0.15.2 peft==0.13.2 bitsandbytes==0.45.0 accelerate==0.34.2
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q wandb sentence-transformers rouge-score nltk

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# W&B login
import os
import wandb

# Set your W&B API key in Kaggle Secrets as WANDB_API_KEY
wandb.login(key=os.environ.get("WANDB_API_KEY"))

run = wandb.init(
    project="gxp-llm",
    name="qwen2.5-7b-qlora-rank16",
    config={
        "model": "unsloth/Qwen2.5-7B",
        "method": "QLoRA (4-bit NF4)",
        "rank": 16,
        "alpha": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "max_seq_length": 2048,
        "batch_size": 2,
        "grad_accum": 4,
        "learning_rate": 2e-4,
        "max_steps": 300,
    }
)

In [ ]:
# Load and prepare data
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("/kaggle/input/gxp-data/train.jsonl")
eval_data = load_jsonl("/kaggle/input/gxp-data/eval.jsonl")

def format_chatml(ex):
    return {"text": "<|im_start|>system\n" + ex["messages"][0]["content"] + "<|im_end|>\n<|im_start|>user\n" + ex["messages"][1]["content"] + "<|im_end|>\n<|im_start|>assistant\n" + ex["messages"][2]["content"] + "<|im_end|>"}

train_ds = Dataset.from_list([format_chatml(ex) for ex in train_data])
eval_ds = Dataset.from_list([format_chatml(ex) for ex in eval_data])

print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

In [ ]:
# Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

FastLanguageModel.for_inference(model)  # Enable faster inference for eval

In [ ]:
# Training
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir="./output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=300,
        learning_rate=2e-4,
        logging_steps=10,
        eval_steps=50,
        save_steps=50,
        report_to="wandb",
        bf16=True,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

trainer.train()

In [ ]:
# Save merged model (16-bit) and LoRA adapter
model.save_pretrained_merged("./merged_16bit", tokenizer, save_method="merged_16bit")
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")

# Also save 4-bit merged for quantization
model.save_pretrained_merged("./merged_4bit", tokenizer, save_method="merged_4bit")

print("Saved: merged_16bit/, merged_4bit/, lora_adapter/")

In [ ]:
# Quick eval on holdout
from eval.run import run_full_eval

results = run_full_eval(
    model_path="./merged_16bit",
    data_dir="/kaggle/input/gxp-data",
    output_dir="./eval_results"
)

In [ ]:
# Upload artifacts to W&B
import wandb
artifact = wandb.Artifact("qwen2.5-7b-gxp-qlora", type="model")
artifact.add_dir("./merged_16bit")
artifact.add_dir("./lora_adapter")
wandb.log_artifact(artifact)

wandb.finish()